In [4]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain.agents import create_agent


C:\Users\HP\AppData\Local\Temp\ipykernel_15392\2177655340.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [20]:
load_dotenv()  # Load environment variables from .env file

True

In [21]:
groq_key = os.getenv("GROQ_API_KEY")
jina_key = os.getenv("JINA_API_KEY")

DATA LOADING

In [22]:
DATA_FILE_PATH = os.path.join("data", "hr_policy.txt")
print("Data file path:", DATA_FILE_PATH)

Data file path: data\hr_policy.txt


DATA INGESTION

In [23]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")
documents = loader.load()
print(documents)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDuring probati

In [24]:
print (len(documents[0].page_content))

2597


chunkking


In [25]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = text_splitter.split_documents(documents)

print(f"Number of chunks: {len(chunks)}")
print(chunks[3].page_content)

Number of chunks: 9
3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Performance is reviewed at the end of the probation period to confirm employment.


Embeddings


In [26]:
embedding_model = JinaEmbeddings(model_name = "jina-embeddings-v4")


Vector DB


In [27]:
vectorstore = FAISS.from_documents(chunks, embedding_model)

print(f"Number of vectors in the vectorstore: {vectorstore.index.ntotal}")

Number of vectors in the vectorstore: 9


In [28]:
print("Total vectors:", vectorstore.index.ntotal)
print("Vector dimensions:", vectorstore.index.d)

for i, doc_id in vectorstore.index_to_docstore_id.items():

    print(f"\n{'='*50}")
    print(f"VECTOR INDEX: {i}")

    vector = vectorstore.index.reconstruct(i)
    print("Embedding:", vector)

    document = vectorstore.docstore.search(doc_id)
    print("Original Text:", document.page_content)
    print("Metadata:", document.metadata)

Total vectors: 9
Vector dimensions: 2048

VECTOR INDEX: 0
Embedding: [-0.0071106  -0.03686523  0.02563477 ...  0.01965332  0.02783203
 -0.00744629]
Original Text: COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)
Metadata: {'source': 'data\\hr_policy.txt'}

VECTOR INDEX: 1
Embedding: [-0.01220703 -0.05078125  0.03222656 ...  0.00411987  0.01818848
 -0.02429199]
Original Text: 1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
Metadata: {'source': 'data\\hr_policy.txt'}

VECTOR INDEX: 2
Embedding: [-0.00402832 -0.05932617  0.00939941 ...  0.01843262  0.02697754
 -0.01843

In [29]:
test_query = "how many sick leave employee get?"

top_matches = vectorstore.similarity_search(test_query, k=2)

print(f"Top {len(top_matches)} matches for the query '{test_query}':")
for i, match in enumerate(top_matches):
    print(f"\nMatch {i + 1}:")
    print(match.page_content)
    print()


Top 2 matches for the query 'how many sick leave employee get?':

Match 1:
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.


Match 2:
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



TOOL


In [36]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def search_hr_policy(questions:str)-> str:
    """
    Search the HR policy documents for relevant information based on the provided questions.
    """
    matching_chunks = retriever.invoke(questions)
    return "\n\n ".join([chunk.page_content for chunk in matching_chunks])

## Retrival

LLM


In [31]:


llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.1,
)
llm.model_name

'openai/gpt-oss-120b'

In [32]:
test_response = llm.invoke("what color is the sky?")
print("Test response:", test_response.content)

Test response: The sky is most often described as **blue** during the day, due to the scattering of sunlight by Earth’s atmosphere. At sunrise and sunset it can take on shades of red, orange, pink, and purple, while a clear night sky appears dark (black) dotted with stars. Cloud cover, weather conditions, and atmospheric particles can also give the sky various hues such as gray, white, or even greenish tones during severe storms.


AI AGENT

In [37]:
hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt = """
    
    You are a friendly hr assiatent
    Always use the search_hr_policy_tool to look up information.
    facts before answering any questions. 
    If it isn't in the search results, say "I don't know" instead of making up an answer instead of making up an answer. 
"""
)

In [38]:
response = hr_assistant.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "how many sick leave employee get?"
            }
        ]
    }
)

In [39]:
print(response["messages"][-1].content)

Employees are entitled to **10 paid sick days per year**.
